#Project
In this project, we use publicly available e-commerce transaction data and supervised machine learning techniques to build a model that predicts which customers are most likely to become repeat purchasers. This will help WitOmni identify high-value customer segments, prioritize marketing efforts, and develop more effective customer retention strategies.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
import tensorflow.keras as keras
from sklearn.preprocessing import StandardScaler
import time

In [ ]:
pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
online_retail = fetch_ucirepo(id=352)

# data (as pandas dataframes)
X = online_retail.data.features
y = online_retail.data.targets

# metadata
print(online_retail.metadata)

# variable information
print(online_retail.variables)


{'uci_id': 352, 'name': 'Online Retail', 'repository_url': 'https://archive.ics.uci.edu/dataset/352/online+retail', 'data_url': 'https://archive.ics.uci.edu/static/public/352/data.csv', 'abstract': 'This is a transactional data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.', 'area': 'Business', 'tasks': ['Classification', 'Clustering'], 'characteristics': ['Multivariate', 'Sequential', 'Time-Series'], 'num_instances': 541909, 'num_features': 6, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': None, 'index_col': ['InvoiceNo', 'StockCode'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2015, 'last_updated': 'Mon Oct 21 2024', 'dataset_doi': '10.24432/C5BW33', 'creators': ['Daqing Chen'], 'intro_paper': {'ID': 361, 'type': 'NATIVE', 'title': 'Data mining for the online retail industry: A case study of RFM model-based customer segmenta

Since the dataset does not come with a predefined target variable (`y`), we need to construct it based on the problem statement. Our goal is to predict 'repeat purchasers'. First, let's create a complete DataFrame by combining the features (`X`) with the ID columns (`InvoiceNo`, `StockCode`) which were separated.

In [ ]:
invoice_stock_ids = online_retail.data.ids
df = pd.concat([invoice_stock_ids, X], axis=1)

display(df.head())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


Now that we have the full dataframe, we can define the target variable `y`. A 'repeat purchaser' can be defined as a customer who has made more than one purchase. We can identify these customers by counting the number of unique `InvoiceNo` per `CustomerID`.

In [ ]:
# Calculate the number of unique invoices per customer
num_invoices_per_customer = df.groupby('CustomerID')['InvoiceNo'].nunique()

# Define repeat purchasers (those with more than 1 invoice)
repeat_purchasers = num_invoices_per_customer[num_invoices_per_customer > 1].index

# Create the target variable 'y': 1 if a repeat purchaser, 0 otherwise
df['is_repeat_purchaser'] = df['CustomerID'].isin(repeat_purchasers).astype(int)

# Display the first few rows with the new target variable and its distribution
display(df[['CustomerID', 'is_repeat_purchaser']].drop_duplicates().head())
print(df['is_repeat_purchaser'].value_counts())
print(f"Percentage of repeat purchasers: {df['is_repeat_purchaser'].mean() * 100:.2f}%")

,CustomerID,is_repeat_purchaser
0,17850.0,1
9,13047.0,1
26,12583.0,1
46,13748.0,1
65,15100.0,1


is_repeat_purchaser
1    378014
0    163895
Name: count, dtype: int64
Percentage of repeat purchasers: 69.76%


In [ ]:
# Now, we can assign the newly created target variable to 'y'
y = df['is_repeat_purchaser']

# Let's check the shape of y now
print(y.shape)

(541909,)


In [ ]:
print (X.shape)

(541909, 6)


In [ ]:
print(y.shape)

(541909,)


Calculating a new variable for creating a new feature, "average order value"

In [ ]:
#How much is being spent
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

In [ ]:
# Determining how much a customer is spending in total + number of orders
customer_avg_order = df.groupby("CustomerID").agg(
    total_spent = ("TotalPrice", "sum"),
    num_orders=("InvoiceNo", "nunique")
)

In [ ]:
# Calculating the average order value feature
customer_avg_order["AverageOrderValue"] = (
    customer_avg_order["total_spent"]/ customer_avg_order["num_orders"]
)

Merging the new feature with the original dataset.

In [ ]:
df = df.merge(
    customer_avg_order[["AverageOrderValue"]],
    on="CustomerID",
    how="left"
)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_repeat_purchaser,TotalPrice,AverageOrderValue_x,AverageOrderValue_y
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,1,15.30,151.103714,151.103714
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,1,20.34,151.103714,151.103714
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,1,22.00,151.103714,151.103714
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,1,20.34,151.103714,151.103714
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,1,20.34,151.103714,151.103714


In [ ]:
list(df.columns)

['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country',
 'is_repeat_purchaser',
 'TotalPrice',
 'AverageOrderValue']

In [ ]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,...,Country,is_repeat_purchaser,TotalPrice,AverageOrderValue_x,AverageOrderValue_y
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,...,United Kingdom,1,15.30,151.103714,151.103714
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,...,United Kingdom,1,20.34,151.103714,151.103714
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,...,United Kingdom,1,22.00,151.103714,151.103714
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,...,United Kingdom,1,20.34,151.103714,151.103714
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,...,United Kingdom,1,20.34,151.103714,151.103714
